# Conjugate Gradient — Interactive Experiments

Test both real symmetric positive-definite and complex Hermitian positive-definite systems against SciPy CG. 

For a real SPD system, CG minimizes the quadratic function

$$\displaystyle
f(x)=\frac12 x^TAx-b^Tx.
$$

For complex Hermitian positive-definite systems,
the corresponding real-valued objective is

$$\displaystyle
f(x)=\frac12 x^HAx-\operatorname{Re}(b^Hx).
$$

The numerical experiments investigate how matrix conditioning
and preconditioning influence convergence.

Explore five inverse-action preconditioner settings.

Select settings and click **Run experiment**. All matrices use a fixed random seed for reproducibility.


In [2]:
from pathlib import Path
import sys
import time

# Find the repository root when launched locally or with Binder.
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Start Jupyter from within the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.sparse.linalg import cg as scipy_cg

from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.notebooks.notebook_utils import (
    spd_problem,
    hermitian_pd_problem,
    block_jacobi_inverse,
    residual_plot,
    error_plot,
    report,
)


## Real SPD and complex Hermitian positive-definite matrices

The complex case replaces transposes with conjugate transposes:
$A=A^H$ and $z^H A z>0$ for every nonzero complex vector $z$.
CG uses $\langle u,v\rangle=u^H v$, and its energy norm is

$$\|e\|_A=\sqrt{e^H A e}.$$

Both test matrices are built with a prescribed positive spectrum. The solver
requires a **fixed Hermitian positive-definite inverse action** for PCG.
The custom example is block-Jacobi: it solves independent principal diagonal
blocks, using conjugate-transpose triangular solves in the complex case.
Exact Cholesky factors the entire matrix, so it is a direct-solve benchmark
(often one PCG iteration), **not** a fair low-cost preconditioner. Comparisons
report true residuals, forward errors, and CG iteration counts; setup costs are
not included in those counts.


## Interactive experiment

The curves show the *true* residual $\|b-Ax_k\|_2$. The dashed line is the absolute/relative stopping threshold.

In [5]:
kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
condition = widgets.FloatLogSlider(
    value=1e4, base=10, min=1, max=7, step=0.5, description="Cond(A)"
)
tol = widgets.FloatLogSlider(
    value=1e-9, base=10, min=-12, max=-5, step=1, description="rtol"
)
budget = widgets.IntSlider(
    value=500, min=20, max=1500, step=20, description="Max iterations"
)
preconds = widgets.Dropdown(
    options=["none", "jacobi", "ssor", "cholesky", "ic0", "custom (block Jacobi)"],
    value="none",
    description="Precond",
    layout=widgets.Layout(width="340px"),
)
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        A, b, x_true = (
            spd_problem(size.value, condition.value)
            if kind.value == "real"
            else hermitian_pd_problem(size.value, condition.value)
        )
        descriptor = (
            "real symmetric positive definite"
            if kind.value == "real"
            else "complex Hermitian positive definite"
        )
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvalsh(A)

        print("SYSTEM INFORMATION")
        print(f"Matrix A: dense {descriptor} ({size.value} × {size.value})")
        print(f"Hermitian error ||A-Aᴴ||₂: {np.linalg.norm(A - A.conj().T, ord=2):.2e}")
        print(f"Spectral interval: [{eigs[0]:.3e}, {eigs[-1]:.3e}]")
        print(f"2-norm condition number: {np.linalg.cond(A):.3e}")
        print(f"||b||₂ = {np.linalg.norm(b):.3e}; ||x_exact||₂ = {np.linalg.norm(x_true):.3e}")
        print(f"Initial x₀ = 0; ||r₀||₂ = {np.linalg.norm(b):.3e}")
        print(f"Stop when ||b-Ax||₂ ≤ max(1e-12, rtol ||b||₂) = {threshold:.3e}")
        print(f"Iteration budget = {budget.value}\n")

        histories = {}
        error_histories = {}
        timings = {}
        for name in (preconds.value,):
            if name == "ic0" and kind.value == "complex":
                print("Skipping IC(0): available only for real SPD matrices.\n")
                continue

            custom = name == "custom (block Jacobi)"
            preconditioner_name = "custom" if custom else None if name == "none" else name

            # Custom setup is external; built-in preconditioners are built
            # inside solve_conjugate_gradient and counted in solver-call time.
            setup_start = time.perf_counter()
            inverse_action = block_jacobi_inverse(A) if custom else None
            external_setup_time = time.perf_counter() - setup_start

            solve_start = time.perf_counter()
            result = solve_conjugate_gradient(
                A, b, rtol=tol.value, atol=1e-12,
                max_iter=budget.value,
                preconditioner=preconditioner_name,
                M=inverse_action,
                store_history=True,
            )
            solver_call_time = time.perf_counter() - solve_start
            timings[name] = (external_setup_time, solver_call_time)

            report(
                "Our CG: " + name, result.solution, x_true, b, A,
                result.iterations, result.converged,
            )
            error = result.solution - x_true
            energy_error = np.sqrt(max(0.0, np.vdot(error, A @ error).real))
            relative_residual = np.linalg.norm(b - A @ result.solution) / np.linalg.norm(b)
            print(f"  Relative residual: {relative_residual:.3e}")
            print(f"  Energy-norm error: {energy_error:.3e}")
            print(f"  External setup time: {external_setup_time:.6f} s")
            print(f"  Solver call time: {solver_call_time:.6f} s")
            print(f"  Status: {result.message}\n")

            histories["CG / " + name] = result.residual_norms
            error_histories["CG / " + name] = [
                np.linalg.norm(xk - x_true) / np.linalg.norm(x_true)
                for xk in result.history
            ]

        # Compare against SciPy CG, recording both metrics at each iteration.
        scipy_history = [np.linalg.norm(b)]
        scipy_errors = [1.0]  # x0 = 0, hence relative error = 1.

        def callback(xk):
            scipy_history.append(np.linalg.norm(b - A @ xk))
            scipy_errors.append(np.linalg.norm(xk - x_true) / np.linalg.norm(x_true))

        scipy_start = time.perf_counter()
        x, info = scipy_cg(
            A, b, rtol=tol.value, atol=1e-12,
            maxiter=budget.value, callback=callback,
        )
        scipy_time = time.perf_counter() - scipy_start
        report("SciPy CG", x, x_true, b, A, len(scipy_history) - 1, info == 0)
        print(f"  Solver call time: {scipy_time:.6f} s")
        print(f"  SciPy info={info} (0 means converged).\n")
        histories["SciPy CG"] = scipy_history
        error_histories["SciPy CG"] = scipy_errors

        residual_plot(
            histories, threshold,
            title="CG — Residual convergence (" + descriptor + ")",
        )
        error_plot(
            error_histories,
            title="CG — Relative solution-error convergence",
        )

        if preconds.value == "cholesky":
            print("Note: Exact Cholesky factors A; it is a direct-solve reference, "
                  "not a cheap preconditioner for large systems.")
        if preconds.value == "custom (block Jacobi)":
            print("Custom inverse action: SPD diagonal blocks of size ≤ 4, "
                  "solved using triangular systems.")
        print("Timing note: built-in preconditioner setup is included in solver-call time; "
              "only custom block-Jacobi has separately measured setup time.")


button.on_click(run)
display(widgets.VBox([kind, size, condition, tol, budget, preconds, button, output]))


### Notes

Small residual does not always imply small solution error for ill-conditioned systems. Compare both metrics; a preconditioner changes convergence but should not change the exact solution.


## Numerical experiment: a small residual can hide a large solution error

For an invertible matrix, $e=x^*-x=A^{-1}(b-Ax)$, hence

$$\|e\|_2\leq\|A^{-1}\|_2\,\|b-Ax\|_2.$$

Construct $A=\mathrm{diag}(1,1/\kappa)$, $x^*=(1,1)^T$, and the initial
guess $x_0=(1,0)^T$. Then the residual is only $1/\kappa$, but the
solution error is $1$ **regardless of** $\kappa$. The interactive test
shows how an absolute stopping threshold can accept a substantially
incorrect solution, even when $\|b\|\approx 1$.


In [8]:
kappa_slider = widgets.FloatLogSlider(value=1e10, base=10, min=2, max=12,
                                               step=1, description="Cond(A)")
absolute_tol = widgets.FloatLogSlider(value=1e-8, base=10, min=-12, max=-3,
                                      step=1, description="atol")
ill_button = widgets.Button(description="Explore residual vs error", button_style="info")
ill_output = widgets.Output()

def explore(_):
    with ill_output:
        clear_output(wait=True)
        kappa = kappa_slider.value
        A = np.diag([1.0, 1.0 / kappa])
        exact = np.ones(2)
        b = A @ exact
        x0 = np.array([1.0, 0.0])
        r0 = b - A @ x0
        initial_error = np.linalg.norm(exact - x0)
        result = solve_conjugate_gradient(A, b, x0=x0, rtol=0.0,
                                          atol=absolute_tol.value, max_iter=2)
        final_error = np.linalg.norm(exact - result.solution)
        final_residual = np.linalg.norm(b - A @ result.solution)
        print(f"A = diag(1, 1/{kappa:.1e}); cond₂(A) = {np.linalg.cond(A):.1e}")
        print(f"Exact x*={exact}; initial x₀={x0}; b={b}")
        print(f"Initial ||r₀||₂={np.linalg.norm(r0):.3e}, ||e₀||₂={initial_error:.3e}")
        print(f"Selected absolute tolerance = {absolute_tol.value:.3e}")
        print(f"CG iterations = {result.iterations}; converged = {result.converged}")
        print(f"Final ||b-Ax||₂={final_residual:.3e}; final ||x*-x||₂={final_error:.3e}")
        print("Compare ||e||₂ to the bound ||A⁻¹||₂ ||r||₂ =",
              f"{kappa * final_residual:.3e}")
        labels = ["Residual norm", "Solution error", "Absolute tolerance"]
        values = [max(final_residual, np.finfo(float).tiny),
                  max(final_error, np.finfo(float).tiny), absolute_tol.value]
        fig, ax = plt.subplots(figsize=(7, 3.7))
        ax.bar(labels, values, color=["#3182bd", "#e6550d", "#777777"])
        ax.set_yscale("log")
        ax.set_ylabel("Magnitude (log scale)")
        ax.set_title("Small residual ≠ small forward error")
        ax.tick_params(axis="x", rotation=10)
        fig.tight_layout()
        plt.show()

ill_button.on_click(explore)
display(widgets.VBox([kappa_slider, absolute_tol, ill_button, ill_output]))


**Interpretation:** Changing the condition number changes how strongly a
residual can underestimate the error. With a loose absolute tolerance the
initial guess may be accepted without a CG step. Tighten `atol` below
$1/\kappa$ to require correction. For SPD systems the error also has an
energy norm, but the residual-to-error relationship still depends on the
spectrum. This deliberately uses a known exact solution and is a diagnostic
example, not evidence of a defect in CG.
